## 1. run the next two cells as-is

In [49]:
#import libraries and set some variables

import requests
import pandas as pd
import xml.etree.ElementTree as ET
from pprint import pprint
from urllib.parse import quote
import numpy as np
import urllib.parse
import os
import warnings
from rapidfuzz import process, fuzz
import nltk
import string
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

#nltk.download('punkt_tab')
#nltk.download('stopwords')

pd.set_option('display.max_columns', None)


#Setting a few Alma Analytics API variables and whatnot
base_path = (r'https://api-na.hosted.exlibrisgroup.com/almaws/v1/analytics/reports?'
             r'limit=1000'
             r'&apikey=l8xxe4f2dc8ebe644d88a84c2ab98b67e6e5'
             r'&path=')  

token_path = (r'https://api-na.hosted.exlibrisgroup.com/almaws/v1/analytics/reports?'
             r'limit=1000'
             r'&apikey=l8xxe4f2dc8ebe644d88a84c2ab98b67e6e5'
             r'&token=')

loc1 = 'shared'
loc2 = 'University of Tennessee, Knoxville'
loc3 = 'Reports'
loc4 = 'Collections'
loc5 = 'Textbook ISBNs for Overlap Report_TM'
if loc5 != '':
    location = '/' + loc1 + '/' + loc2 + '/' + loc3 + '/' + loc4 + '/' + loc5 
else:
    location = '/' + loc1 + '/' + loc2 + '/' + loc3 + '/' + loc4 + '/'

get_path = location 
url_encoded_path = quote(get_path)
report_path = base_path + url_encoded_path

xml_snippet = '''
<sawx:expr xsi:type="sawx:list" op="in"
  xmlns:saw="com.siebel.analytics.web/report/v1.1"
  xmlns:sawx="com.siebel.analytics.web/expression/v1.1" 
  xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
  xmlns:xsd="http://www.w3.org/2001/XMLSchema">

<sawx:expr xsi:type="sawx:sqlExpression">"Bibliographic Details"."MMS Id"</sawx:expr>
{}
</sawx:expr>
'''

In [2]:
report_path

'https://api-na.hosted.exlibrisgroup.com/almaws/v1/analytics/reports?limit=1000&apikey=l8xxe4f2dc8ebe644d88a84c2ab98b67e6e5&path=/shared/University%20of%20Tennessee%2C%20Knoxville/Reports/Collections/Textbook%20ISBNs%20for%20Overlap%20Report_TM'

In [3]:
#define some functions

#function to extract data from the analytics report
def extract_data(row_elements, columns):
    data = []
    for row in row_elements:
        row_data = {}
        for col in row.findall('{urn:schemas-microsoft-com:xml-analysis:rowset}*'):
            column_name = col.tag.split('}')[1]
            column_value = col.text
            if column_name in columns:  # Check if the column is in the columns dictionary
                row_data[columns[column_name]] = column_value if column_value is not None else np.nan  # Insert NaN if value is missing
        data.append(row_data)
    return data

#pulling the analytics report and extracting data
def get_report(response):
    # Parse the XML response
    root = ET.fromstring(response.content)
    # Extract the ResumptionToken and IsFinished elements from the response XML
    resumption_token_element = root.find('.//ResumptionToken')
    resumption_token = resumption_token_element.text if resumption_token_element is not None else None
    is_finished_element = root.find('.//IsFinished')
    if is_finished_element is not None:
        is_finished = is_finished_element.text
    else:
        is_finished = None

    namespaces = {
        'xmlns': 'urn:schemas-microsoft-com:xml-analysis:rowset',
        'xsd': 'http://www.w3.org/2001/XMLSchema',
        'saw-sql': 'urn:saw-sql'
    }
    columns = {}

    for i, col in enumerate(root.findall('.//xsd:element[@saw-sql:columnHeading]', namespaces)):
        columns[f'Column{i}'] = col.get('{urn:saw-sql}columnHeading', col.get('name'))
    
    # Extract the data from the XML and store it in a list of dictionaries
    all_data = []
    data = extract_data(root.findall('.//{urn:schemas-microsoft-com:xml-analysis:rowset}Row'), columns)
    all_data.extend(data)
    
    # Iterate through subsequent GET requests until IsFinished is True
    while is_finished == 'false':
        # Make the next GET request with the ResumptionToken
        response = requests.get(token_path + resumption_token)

        # Extract the ResumptionToken and IsFinished elements from the response XML
        root = ET.fromstring(response.content)
        is_finished = root.find('.//IsFinished').text

        # Process the response data
        data = extract_data(root.findall('.//{urn:schemas-microsoft-com:xml-analysis:rowset}Row'), columns)
        all_data.extend(data)

    report_df = pd.DataFrame(all_data)

    # Drop the first column of the dataframe
    if not report_df.empty:
        report_df = report_df.drop(report_df.columns[0], axis=1)

    # Reset the index of the dataframe
    report_df = report_df.reset_index(drop=True)

    return report_df


def get_and_concat_report(report_path, report_data_df):
    response = requests.get(report_path)
    report_df = get_report(response)
    print(len(report_df))
    return pd.concat([report_data_df, report_df], ignore_index=True)

# Define a custom function to concatenate non-NaN values and deduplicate the notes fields which contain information about 
# online access type for each ebook
def concatenate_and_deduplicate_notes(row):
    notes = [str(row['Portfolio Public Note']), str(row['Portfolio Authentication Note']), str(row['Electronic location and access'])]
    non_nan_notes = set(note for note in notes if note != 'nan')
    return '; '.join(non_nan_notes)


## 2. Set the date and location of important Excel files

In [4]:
#set the date string for the rest of this notebook
date_str = '2026-11-20'

#Set the location where all bookstore data will be saved to and read from for the rest of this notebook

bookstoreFolder = r"C:\Users\aharrima\GitHub_UTK\Bookstore_Program_Holding_Area"

directory = os.chdir(bookstoreFolder)

## 3. Read in the searchresults file from the bookstore
## make sure to point read_excel to the correct file location!

In [5]:
#load most recent file from bookstore
# you will need to have saved the automated bookstore file using format "searchresults_YYYY-MM-DD.xlsx" to your computer
df = pd.read_excel(r'searchresults_'+date_str+'.xlsx')

# Preview the sorted DataFrame
df


,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity
0,SPRING 2027,ACCOUNTING,202720-ACCT.303.001-41046,"Short, Justin",9781394254439,INTERMEDIATE ACCOUNTING,"KIESO, DONALD E., WEYGANDT, JERRY J., WARFIELD...","WILEY & SONS, INCORPORATED, JOHN",57
1,SPRING 2027,ACCOUNTING,202720-ACCT.303.002-41047,"Short, Justin",9781394254439,INTERMEDIATE ACCOUNTING,"KIESO, DONALD E., WEYGANDT, JERRY J., WARFIELD...","WILEY & SONS, INCORPORATED, JOHN",57
2,SPRING 2027,ACCOUNTING,202720-ACCT.303.003-41048,"Short, Justin",9781394254439,INTERMEDIATE ACCOUNTING,"KIESO, DONALD E., WEYGANDT, JERRY J., WARFIELD...","WILEY & SONS, INCORPORATED, JOHN",30
3,SPRING 2027,ACCOUNTING,202720-ACCT.303.004-41017,"Ereddia, Laurie",9781394246144,INTERMEDIATE ACCOUNTING,"KIESO, DONALD E., WEYGANDT, JERRY J., WARFIELD...","WILEY & SONS, INCORPORATED, JOHN",60
4,SPRING 2027,ACCOUNTING,202720-ACCT.303.005-41049,"Clapp, Randall",9781394246144,INTERMEDIATE ACCOUNTING,"KIESO, DONALD E., WEYGANDT, JERRY J., WARFIELD...","WILEY & SONS, INCORPORATED, JOHN",60
...,...,...,...,...,...,...,...,...,...
2215,SPRING 2027,"WOMEN, GENDER AND SEXUALITY",202720-WGS.332.003-30902,"Nicks, Robin",9781250766571,FIREKEEPER'S DAUGHTER,"BOULLEY, ANGELINE",SQUARE FISH,10
2216,SPRING 2027,"WOMEN, GENDER AND SEXUALITY",202720-WGS.332.003-30902,"Nicks, Robin",9780307758071,HERLAND,"PERKINS GILMAN, CHARLOTTE AND GILMAN, CHARLOTT...",KNOPF DOUBLEDAY PUBLISHING GROUP,10
2217,SPRING 2027,"WOMEN, GENDER AND SEXUALITY",202720-WGS.332.003-30902,"Nicks, Robin",9780525620792,MEXICAN GOTHIC,"MORENO-GARCIA, SILVIA AND MORENO-GARCIA, SILVIA",RANDOM HOUSE WORLDS,10
2218,SPRING 2027,"WOMEN, GENDER AND SEXUALITY",202720-WGS.454.001-41362,"Presser, Lois",9780465098736,TRAUMA AND RECOVERY,JUDITH LEWIS HERMAN,HACHETTE,30


## 4. do some work on the bookstore-provided file to generate a list of ISBNs that 
## we will use in a later step

In [6]:
#create list of ISBNs for use in Alma Set creation
#do some pre-processing on the data for cleanup purposes
isbns = df[['ISBN']]
isbns = isbns.dropna()
isbns = isbns.drop_duplicates()
isbns['ISBN'] = isbns['ISBN'].apply(lambda x: x.replace('-',''))
isbns = isbns.astype(str)
#isbns = isbns[(isbns['ISBN'].str.len() == 13) & (isbns['ISBN'].str.isdigit())]
isbns = isbns[(isbns['ISBN'].str.isdigit())]
isbns = isbns.reset_index()
isbns = isbns.drop('index', axis = 1)
isbns = isbns.iloc[isbns['ISBN'].str.len().sort_values(ascending = False).index]
isbns

,ISBN
525,9780367740795
0,9781394254439
1,9781394246144
2,9780761181361
3,9798765778760
...,...
127,780134838755
366,1259192024
469,1266821260
410,97839315


In [7]:
isbns.dtypes

ISBN    str
dtype: object

## Fix Known Mismatched ISBNs in the cell below.

These titles can be possibly identified with the fuzzy title matching section (Section #7 later in this code), or by checking their Alma holding and VolBooks title information against each other manually.

In [60]:
#Fix known mismatching ISBNs

for i in range(isbns.shape[0]):
    if isbns.at[i, 'ISBN'] == '9780191037627': #Silas Marner
        print('Found Silas Marner')
        isbns.at[i, 'ISBN'] = '9780191037610'    
    if isbns.at[i, 'ISBN'] == '9781040675892': #Sick Enough
        print('Found Sick Enough')       
        isbns.at[i, 'ISBN'] = '9781041036487'   
    if isbns.at[i, 'ISBN'] == '9781440855375': #Evaluation and Measurement of Library Services
        print('Found Evaluation and Measurement of Library Services')     
        isbns.at[i, 'ISBN'] = '9781440855368'     
    if isbns.at[i, 'ISBN'] == '9781003155119': #Information Science: The Basics
        print('Found Information Science')     
        isbns.at[i, 'ISBN'] = '9780367725204'      
    if isbns.at[i, 'ISBN'] == '9781119628149': #International Financial Statement Analysis
        print('Found International Financial Statement Analysis')    
        isbns.at[i, 'ISBN'] = '9781119682141'     
    if isbns.at[i, 'ISBN'] == '9781351133135': #Counseling Children and Adolescents
        print('Found Counseling Children and Adolescents')  
        isbns.at[i, 'ISBN'] = '9781351133159'   
    if isbns.at[i, 'ISBN'] == '9781501505935': #Demand and Supply Integration
        print('Found Demand and Supply Integration')  
        isbns.at[i, 'ISBN'] = '9781501506024'  
    if isbns.at[i, 'ISBN'] == '9781506352916': #Performance Appraisal and Management
        print('Found Performance Appraisal and Management')  
        isbns.at[i, 'ISBN'] = '9781506352886'  
    if isbns.at[i, 'ISBN'] == '9781317346005': #On the History of Political Philosophy
        print('Found On the History of Political Philosophy')
        isbns.at[i, 'ISBN'] = '9781317346012'
    if isbns.at[i, 'ISBN'] == '9781000690835': #Child-Centered Play Therapy
        print('Found Child-Centered Play Therapy')
        isbns.at[i, 'ISBN'] = '9781003260431' 
    if isbns.at[i, 'ISBN'] == '9781483315430': #Controversies in Contemporary Advertising
        print('Found Controversies in Contemporary Advertising')
        isbns.at[i, 'ISBN'] = '9781452233130' 
    if isbns.at[i, 'ISBN'] == '9781538139332': #Cooperation and Conflict Between State and Local Government
        print('Found Cooperation and Conflict Between State and Local Government')
        isbns.at[i, 'ISBN'] = '9798216405665' 
    if isbns.at[i, 'ISBN'] == '9781315218489': #Advanced Linear Algebra for Engineers with MATLAB
        print('Found Advanced Linear Algebra for Engineers with MATLAB') 
        isbns.at[i, 'ISBN'] = '9781420095241' 
    if isbns.at[i, 'ISBN'] == '9781351834377': #Advanced Linear Algebra for Engineers with MATLAB, second ISBN
        print('Found Advanced Linear Algebra for Engineers with MATLAB')
        isbns.at[i, 'ISBN'] = '9781420095241'  
    if isbns.at[i, 'ISBN'] == '9780393069754': #Beowulf
        print('Found Beowulf') 
        isbns.at[i, 'ISBN'] = '9781350212718'
    if isbns.at[i, 'ISBN'] == '9781483322100': #Conflict, War, and Peace: An Introduction to Scientific Research
        print('Found Conflict, War, and Peace')
        isbns.at[i, 'ISBN'] = '9781071934111'
    if isbns.at[i, 'ISBN'] == '9780742575622': #Criminologiccal Theory: An Analysis of its Underlying Assumptions
        print('Found Criminological Theory')
        isbns.at[i, 'ISBN'] = '9798216404972'
    if isbns.at[i, 'ISBN'] == '9780191007774': #Global Environments Through the Quarternary: Exploring Environmental Change
        print('Found Global Environments Through the Quarternary')
        isbns.at[i, 'ISBN'] = '9780191810169'
    if isbns.at[i, 'ISBN'] == '9780191606373': #History: A Very Short Introduction
        print('Found History')
        isbns.at[i, 'ISBN'] = '9780191540189'
    if isbns.at[i, 'ISBN'] == '9780124079489': #Introduction to Probability Models
        print('Found Introduction to Probability Models')
        isbns.at[i, 'ISBN'] = '9780125984560'
    if isbns.at[i, 'ISBN'] == '9781948057882': #John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
        print("Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition")
        print('Make sure this is the Unlimited version from ProQuest, not the 1 user from R2 Digital!')
        isbns.at[i, 'ISBN'] = '9781948057899'
    if isbns.at[i, 'ISBN'] == '9780838937952': #Metadata
        print('Found Metadata')
        isbns.at[i, 'ISBN'] = '9780838948637'
    if isbns.at[i, 'ISBN'] == '9780307757890': #Paradise Lost
        print('Found Paradise Lost')
        isbns.at[i, 'ISBN'] = '9781317865735'
    if isbns.at[i, 'ISBN'] == '9781466887787': #14-18: Understanding the Great War
        print('Found 14-18 Understanding the Great War')
        isbns.at[i, 'ISBN'] = '9780809046430'
    if isbns.at[i, 'ISBN'] == '9781400030156': #The Big Sleep
        print('Found The Big Sleep')
        isbns.at[i, 'ISBN'] = '9781479403776'
    if isbns.at[i, 'ISBN'] == '9780191629112': #The Book of Marvels and Travels
        print('Found The Book of Marvels and Travels')
        isbns.at[i, 'ISBN'] = '9780191629105'
    if isbns.at[i, 'ISBN'] == '9780140449617': #The Death of Ivan Ilyich and Other Stories
        print('Found The Death of Ivan Ilyich')
        isbns.at[i, 'ISBN'] = '9780191649042'
    if isbns.at[i, 'ISBN'] == '9781479823550': #Forced Out: Migrant Mothers in Search of Refuge and Hope
        print('Found Forced Out')
        isbns.at[i, 'ISBN'] = '9781479823529'
    if isbns.at[i, 'ISBN'] == '9780198745686': #Imperial Apocalypse: The Great War and the Destruction of the Russian Empire
        print('Found Imperial Apocalypse')
        isbns.at[i, 'ISBN'] = '9781644699522'
    if isbns.at[i, 'ISBN'] == '9780262367509': #Introduction to Algorithms, Fourth Edition
        print('Found Introduction to Algorithms')
        isbns.at[i, 'ISBN'] = '9780262046305'
    if isbns.at[i, 'ISBN'] == '9780813178288': #John Hervey Wheeler, Black Banking, and the Economic Struggle for Civil Rights]
        print('Found John Hervey Wheeler')
        isbns.at[i, 'ISBN'] = '9780813178271'
    if isbns.at[i, 'ISBN'] == '9781452289236': #Mental Health Case Management: A Practical Guide
        print('Found Mental Health Case Management')
        isbns.at[i, 'ISBN'] = '9781452256078'
    if isbns.at[i, 'ISBN'] == '9781101576984': #Merchants of Culture The Publishing Business in the Twenty-First Century
        print('Found Merchants of Culture')
        isbns.at[i, 'ISBN'] = '9780745661421'
    if isbns.at[i, 'ISBN'] == '9781319049966': #Narrative of the Life of Frederick Douglass: An American Slave, Written by Himself
        print('Found Frederick Douglass')
        isbns.at[i, 'ISBN'] = '9780760355466'
    if isbns.at[i, 'ISBN'] == '9780199720149': #The Oxford Handbook of Engineering and Technology in the Classical World
        print('Found Handbook of Engineering and Technology')
        isbns.at[i, 'ISBN'] = '9780195187311'
    if isbns.at[i, 'ISBN'] == '9781350113008': #Performance Drawing
        print('Found Performance Drawing')
        isbns.at[i, 'ISBN'] = '9781350113015'
    if isbns.at[i, 'ISBN'] == '9780191624711': #Persuasion
        print('Found Persuasion')
        isbns.at[i, 'ISBN'] = '9781316848630'
    if isbns.at[i, 'ISBN'] == '9781009314848': #The Gas Mask in Interwar Germany: Visions of Chemical Modernity
        print('Found Gas Mask')
        isbns.at[i, 'ISBN'] = '9781009314862'
    if isbns.at[i, 'ISBN'] == '9780385093309': #Good-Bye to All That
        print('Found Good-Bye to All That')
        isbns.at[i, 'ISBN'] = '9781774649367'
    if isbns.at[i, 'ISBN'] == '9781119837909': #Research Ethics for Scientists
        print('Found Research Ethics for Scientists')
        isbns.at[i, 'ISBN'] = '9781119978862'
    if isbns.at[i, 'ISBN'] == '9780307432568': #The Return of the Soldier
        print('Found The Return of the Soldier')
        isbns.at[i, 'ISBN'] = '9780486846545'
    if isbns.at[i, 'ISBN'] == '9781496203168': #Siberian Exile
        print('Found Siberian Exile')
        isbns.at[i, 'ISBN'] = '9780803299597'
    if isbns.at[i, 'ISBN'] == '9780190879976': #Teaching Music Theory
        print('Found Teaching Music Theory')
        isbns.at[i, 'ISBN'] = '9780197510575'
    if isbns.at[i, 'ISBN'] == '9780197644058': #An Uneasy Peace
        print('Found An Uneasy Peace')
        isbns.at[i, 'ISBN'] = '9780197619407'
    if isbns.at[i, 'ISBN'] == '9781975121549': #Fleisher & Ludwig's Textbook of Pediatric Emergency Medicine
        print('Found Fleisher & Ludwig Textbook')
        isbns.at[i, 'ISBN'] = '9781975121532'
    if isbns.at[i, 'ISBN'] == '9781482542222': #Treating Complex Traumatic Stress Disorders in Adults: Scientific Foundations and Therapeutic Models
        print('Found Treating Complex Traumatic Stress Disorders in Adults')
        isbns.at[i, 'ISBN'] = '9781462542239'
    if isbns.at[i, 'ISBN'] == '9781316044582': #Elements of Mathematical Ecology
        print('Found Elements of Mathematical Ecology')
        isbns.at[i, 'ISBN'] = '9780511608520'
    if isbns.at[i, 'ISBN'] == '9780307758071': #Herland
        print('Found Herland')
        isbns.at[i, 'ISBN'] = '9781667601038'
    if isbns.at[i, 'ISBN'] == '9780393270372': #Narrative of the Life of Frederick Douglass
        print('Found Frederick Douglass')
        isbns.at[i, 'ISBN'] = '9780760355466'
    if isbns.at[i, 'ISBN'] == '9780199748037': #Population Health: Concepts and Methods
        print('Found Population Health')
        isbns.at[i, 'ISBN'] = '9780199864379'
    if isbns.at[i, 'ISBN'] == '9780300048124': #Preschool in Three Cultures: Japan, China and the United States
        print('Found Preschool in Three Cultures')
        isbns.at[i, 'ISBN'] = '9780585358543'
    if isbns.at[i, 'ISBN'] == '9781101078143': #The Souls of BLack Folk
        print('Found The Souls of Black Folk')
        isbns.at[i, 'ISBN'] = '9781315631998'
    if isbns.at[i, 'ISBN'] == '0679601813': #The Brothers Karamazov
        print('Found The Brothers Karamazov')
        isbns.at[i, 'ISBN'] = '9781786565556'
    if isbns.at[i, 'ISBN'] == '9781612495736': #Manufacturing Facilities Design and Material Handling: Sixth Edition
        print('Found Manufacturing Facilities Design')
        isbns.at[i, 'ISBN'] = '9781612495729'
    if isbns.at[i, 'ISBN'] == '9780191605475': #Bleak House
        print('Found Bleak House')
        isbns.at[i, 'ISBN'] = '9780486822556'
    if isbns.at[i, 'ISBN'] == '9781119348450': #Modern Nuclear Chemistry
        print('Found Modern Nuclear Chemistry')
        isbns.at[i, 'ISBN'] = '9780471768623'
    if isbns.at[i, 'ISBN'] == '9781903544877': #Planetary Geology: An Introduction
        print('Found Planetary Geology')
        isbns.at[i, 'ISBN'] = '9781780461045'
    if isbns.at[i, 'ISBN'] == '9783031758539': #R for Political Science Research: An Introduction for Absolute Beginners
        print('Found R for Political Science Research')
        isbns.at[i, 'ISBN'] = '9783031758522'
    if isbns.at[i, 'ISBN'] == '9780674040793': #Clinging to Mammy: The Faithful Slave in Twentieth-Century America
        print('Found Clinging to Mammy')
        isbns.at[i, 'ISBN'] = '9780674024335'
    if isbns.at[i, 'ISBN'] == '9781460404027': #The Return of the Soldier
        print('Found Return of the Soldier')
        isbns.at[i, 'ISBN'] = '9780486846545'
    if isbns.at[i, 'ISBN'] == '9781315365619': #Instrumentation, Measurements, and Experiments in Fluids
        print('Found Experiments in Fluids')
        isbns.at[i, 'ISBN'] = '9780849307591'
    if isbns.at[i, 'ISBN'] == '9781612930428': #Pride and Prejudice
        print('Found Pride and Prejudice')
        isbns.at[i, 'ISBN'] = '9781504033831'
    if isbns.at[i, 'ISBN'] == '9781317615880': #Environmental Ethics for the Long Term
        print('Found Environmental Ethics for the Long Term')
        isbns.at[i, 'ISBN'] = '9781317615897'
    if isbns.at[i, 'ISBN'] == '9781119943501': #The Eyes of the Skin
        print('Found The Eyes of the Skin')
        isbns.at[i, 'ISBN'] = '9781394200702'
    if isbns.at[i, 'ISBN'] == '9780806130576': #Horace Epodes and Odes
        print('Found Horace Epodes and Odes')
        isbns.at[i, 'ISBN'] = '0585165475'
    if isbns.at[i, 'ISBN'] == '9781119585473': #Nonprofit Management 101
        print('Found Nonprofit Management 101')
        isbns.at[i, 'ISBN'] = '9781119585527'
    if isbns.at[i, 'ISBN'] == '9780062466679': #A People's History of the United States
        print("Found A People's History of the United States")
        isbns.at[i, 'ISBN'] = '9781317325291'

Found Horace Epodes and Odes
Found The Eyes of the Skin
Found Environmental Ethics for the Long Term
Found A People's History of the United States
Found Nonprofit Management 101


## 5. Save the isbns to an Excel file

In [61]:
#create new itemized set in Alma using this isbn file:
isbns.to_excel(date_str+'_isbns.xlsx', index = False)

# after the set is created, export the results (All Fields) and save the file using format "YYYY-MM-DD_set_output.xlsx"

## Use the isbn file to create an itemized set in alma
## After the set is created, export the results (All Fields) and save using the file name format "YYYY-MM-DD_set_output.xlsx" to the same location that all your other files for this process live in

## 6. The following cell will use the set_output you just made to call on the alma analytics API, looking for any matching ISBNs

In [62]:
#use the alma API and the set results to get all ISBNs for any MMS IDs found in set
#analytics report: Collections folder > "Textbook ISBNs for Overlap Report_TM"

#create list of mms_ids from set output
mms_ids = pd.read_excel(date_str+"_set_output.xlsx")
#mms_ids = pd.read_excel('ProQuest to EBSCO Check 2-4.xlsx')
mms_ids = mms_ids[['MMS ID']]

# Get the list of MMS IDs from the DataFrame
mms_list = mms_ids['MMS ID'].tolist()

# Define the batch size for API call(s)
batch_size = 50

# Create a list to store the batches
batches = []

# Iterate through the MMS ID list and create batches
for i in range(0, len(mms_list), batch_size):
    batch = mms_list[i : i + batch_size]
    batches.append(batch)

#create an empty list into which we will put api call results
df_list = []

#for each batch of mms ids, call analytics api
for batch in batches:
    values_xml = '\n'.join(f'<sawx:expr xsi:type="xsd:string">{value}</sawx:expr>' for value in batch)
    formatted_xml_snippet = xml_snippet.format(values_xml)

    url_encoded_filter_query = urllib.parse.quote(formatted_xml_snippet)

    full_path = base_path + url_encoded_path + r'&filter=' + url_encoded_filter_query

    report_data_df = get_and_concat_report(full_path, pd.DataFrame())
    df_list.append(report_data_df)
    
# concatenate together all the api call results into one big dataframe
combined_df = pd.concat(df_list, ignore_index=True)
combined_df


36
33
20


,Electronic location and access,ISBN (Normalized),ISBN,MMS Id,Title,Electronic Collection Public Name,Availability,Portfolio Id,Portfolio Public Note,No. of Available Portfolio,Portfolio Authentication Note
0,https://search.ebscohost.com/login.aspx?direct...,144434594X; 9781444345940; 1444345915; 9781444...,144434594X; 9781444345940; 1444345915; 9781444...,9922520450102311,A companion to paleopathology /,ProQuest Ebook Central DDA Discovery,Available,53366614400002311,Click to View,0,NaN
1,https://www.taylorfrancis.com/books/9781003183686,1003183689; 9781003183686; 1000811808; 9781000...,1003183689; 9781003183686; 1000811808; 9781000...,9926319485402311,"Deaf people and society : psychological, socio...",Taylor & Francis Ebooks (EBA),Available,53395065450002311,NaN,0,NaN
2,https://www.degruyter.com/isbn/9781613630174 U...,9781613630174; 1613630174; 9781613630181; 1613...,9781613630174; 1613630174; 9781613630181; 1613...,9926828613202311,Financial Literacy for Managers : Finance and ...,De Gruyter Ebooks Individually Purchased,Available,53446301860002311,NaN,0,Unlimited Access
3,https://search.ebscohost.com/login.aspx?direct...,144434594X; 9781444345940; 1444345915; 9781444...,144434594X; 9781444345940; 1444345915; 9781444...,9922520450102311,A companion to paleopathology /,Ebooks moved from ProQuest to EBSCO,Available,53449629450002311,NaN,1,NaN
4,https://ebookcentral.proquest.com/lib/utk/deta...,9780470671078; 0470671076; 9781119950264; 1119...,9780470671078; 0470671076; 9781119950264; 1119...,9926276606602311,A guide to old English,ProQuest Ebook Central: Academic Complete,Available,53391009200002311,NaN,1,NaN
5,https://muse.jhu.edu/book/46850/ Unlimited Acc...,0809326620; 9780809326624; 9780809326631; 0809...,0809326620; 9780809326624; 9780809326631; 0809...,9926062446102311,Appeals in Modern Rhetoric An Ordinary Languag...,Project Muse Ebooks Individually Purchased,Available,53356492330002311,Unlimited Access,1,NaN
6,http://search.ebscohost.com/login.aspx?direct=...,030738862X; 9780307388629; 9780394535975; 0394...,030738862X; 9780307388629; 9780394535975; 0394...,9925919468902311,Beloved /,EBSCO Ebooks Individually Purchased,Available,53331965650002311,NaN,1,Online Access - 1 user
7,http://dx.doi.org/10.1007/978-1-4615-0729-1,9781461507291; 1461507294; 9780306472749; 0306...,9781461507291; 1461507294; 9780306472749; 0306...,9925584156402311,Brewing,Springer English/International Ebook Collection,Available,53263558200002311,NaN,1,NaN
8,https://www.jstor.org/stable/10.2307/j.ctv2rr3...,194483818X; 9781944838188; 9781944838195; 1944...,194483818X; 9781944838188; 9781944838195; 1944...,9926431786402311,"Case studies in deaf education : inquiry, appl...",JSTOR Rialto Implementation,Available,53411436840002311,Unlimited Access,1,NaN
9,NaN,1523140984; 9781523140985; 160595599X; 9781605...,1523140984; 9781523140985; 160595599X; 9781605...,9926398687202311,Construction site erosion and sediment control...,Knovel Civil Engineering & Construction Materi...,Available,53406818060002311,NaN,1,NaN


In [63]:
#combined_df.to_excel('Alma Textbook List Check Summer 2026.xlsx')

In [64]:
combined_df.sort_values(by = 'Title')['Title'].unique()

<ArrowStringArray>
[                                                                                                   'A Pocket Style Manual',
                                                                                          'A companion to paleopathology /',
                                                                                                   'A guide to old English',
                                                                                'A people's history of the United States /',
                      'A practical guide to action research and teacher enquiry : making a difference in the early years /',
                                                               'Appeals in Modern Rhetoric An Ordinary Language Approach /',
 'Artificial intelligence for cybersecurity : develop AI approaches to solve cybersecurity problems in your organization /',
                                                                                                          

## 7. Check for mismatched ISBNs using Fuzzy Matching

Since titles often have several different ISBNs, and we only receive one ISBN from VolBooks, we need to make sure as many of those ISBNs match up as possible. The cell below checks for any of these potential mismatches.
|
### If a title's ISBN does not align between VolBooks and Alma, go to the ISBN fixing cell in Section #4 and edit the VolBooks holdings to match the Alma holdings.

**NOTE:** This code finds the closest matching title within the VolBooks and Alma lists, and thus it may not always be a direct match. Double-check the code before changing any ISBNs.

In [41]:
## Load the Alma Data

volBooksTitleList = df['Short Title'].unique()
almaUnlimited = pd.read_excel('Textbook ISBN Checking File November 20.xlsx')

In [42]:
## Alma ISBN Splitting Code
combinedCopy = almaUnlimited

combinedCopy['ISBN (Normalized) List'] = combinedCopy['ISBN (Normalized)'].apply(lambda x: str(x).split(";"))

expandedCopy = combinedCopy.explode('ISBN (Normalized) List')

expandedCopy['ISBN (Normalized) List'] = expandedCopy['ISBN (Normalized) List'].str.strip()

In [43]:
## Clean and Tokenize the Titles

def TextPreProcess(title):
    title = str(title)
    title = title.lower()
    title = title.translate(str.maketrans('', '', string.punctuation))
    tokens = word_tokenize(title)

    stop_words = set(stopwords.words('english'))
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

## Load Title Lists
df['Tokenized Title'] = df['Short Title'].apply(TextPreProcess)
expandedCopy['Tokenized Title'] = expandedCopy['Title'].apply(TextPreProcess)

In [44]:
## Create Stems and Rejoin Titles

stemmer = SnowballStemmer(language = 'english')

def StemTokens(tokens):
    stems = [stemmer.stem(token) for token in tokens]
    return ' '.join(stems)

df['Stemmed Title'] = df['Tokenized Title'].apply(StemTokens)
expandedCopy['Stemmed Title'] = expandedCopy['Tokenized Title'].apply(StemTokens)

In [45]:
## Initialize Information

volBooksTitleList = df['Stemmed Title'].unique()
almaTitleList = expandedCopy['Stemmed Title'].unique()

almaTitle = pd.Series([])
volBooksTitle = pd.Series([])
almaISBN = pd.Series([])
volBooksISBN = pd.Series([])
almaStem = pd.Series([])
volBooksStem = pd.Series([])

In [50]:
expandedCopy.columns

Index(['No. of Available Portfolio', 'Electronic location and access', 'Title',
       'MMS Id', 'Portfolio Id', 'Electronic Collection Public Name',
       'Portfolio Public Note', 'Portfolio Authentication Note',
       'Service Authentication Note', 'ISBN', 'ISBN (Normalized)',
       'Availability', 'Portfolio Activation Date', 'ISBN (Normalized) List',
       'Tokenized Title', 'Stemmed Title'],
      dtype='str')

In [47]:
## Find the Potential Titles with Mismatched ISBNs

## Note: this code chunk can take many minutes to run to completion

for i in range(len(volBooksTitleList)):
    titleName = volBooksTitleList[i]
    bestResult = process.extractOne(titleName, almaTitleList, scorer = fuzz.token_sort_ratio)
    
    selectVolBooksRows = df[df['Stemmed Title'] == titleName]
    volBooksStem = pd.concat([volBooksStem, pd.Series([selectVolBooksRows['Stemmed Title'].iloc[0]])], ignore_index = True)
    volBooksISBN = pd.concat([volBooksISBN, pd.Series([selectVolBooksRows['ISBN'].iloc[0]])], ignore_index = True)
    volBooksTitle = pd.concat([volBooksTitle, pd.Series([selectVolBooksRows['Short Title'].iloc[0]])], ignore_index = True)
    
    selectAlmaRows = expandedCopy[expandedCopy['Stemmed Title'] == bestResult[0]]
    almaStem = pd.concat([almaStem, pd.Series([selectAlmaRows['Stemmed Title'].iloc[0]])], ignore_index = True)
    almaISBN = pd.concat([almaISBN, pd.Series([selectAlmaRows['ISBN (Normalized) List'].iloc[0]])], ignore_index = True)
    almaTitle = pd.concat([almaTitle, pd.Series([selectAlmaRows['Title'].iloc[0]])], ignore_index = True)
    
    for j in range(expandedCopy.shape[0]):
        if expandedCopy['Title'].iloc[j] == bestResult:
            almaISBN = pd.concat(volBooksISBN, pd.Series([expandedCopy.at[j, 'ISBN (Normalized) List']]), ignore_index = True)
            almaTitle = pd.concat(volBooksTitle, pd.Series([bestResult]), ignore_index = True)
            next
    for k in range(df.shape[0]):
        if expandedCopy['Title'].iloc[k] == titleName:
            volBooksISBN = pd.concat(almaISBN, pd.Series([df.at[k, 'ISBN']]), ignore_index = True)
            volBooksTitle = pd.concat(almaTitle, pd.Series([titleName]), ignore_index = True)
            next

titleComparisonTable = pd.concat([volBooksStem, almaStem, volBooksISBN, almaISBN, volBooksTitle, almaTitle], axis = 1)
    
titleComparisonTable.columns = ['VolBooks Stemmed Title', 'Alma Stemmed Title', 'VolBooks ISBN', 'Alma ISBN', 'VolBooks Raw Title', 'Alma Raw Title']

titleComparisonTable

,VolBooks Stemmed Title,Alma Stemmed Title,VolBooks ISBN,Alma ISBN,VolBooks Raw Title,Alma Raw Title
0,intermedi account,comic intermedi,9781394254439,1496863143,INTERMEDIATE ACCOUNTING,Comics and intermediality /
1,show work 10 way share creativ get discov,last human job work connect disconnect world,9780761181361,9780691243771,SHOW YOUR WORK! : 10 WAYS TO SHARE YOUR CREATI...,The last human job : the work of connecting in...
2,advertis principlescustom,design principl,9798765778760,9781616167356,ADVERTISING PRINCIPLES-CUSTOM,Design. The Principles /
3,advertis concept book think design later,narrat advertis model conceptu digit age,9780500292679,9781522523741,"THE ADVERTISING CONCEPT BOOK : THINK NOW, DESI...",Narrative Advertising Models and Conceptualiza...
4,art pitch,part speech,9781137512338,9780736556880,THE ART OF THE PITCH,Parts of speech.
...,...,...,...,...,...,...
501,requiem por un campesino espanol stockcero,lui valdez el teatro campesino,9789871136483,9781421338668,REQUIEM POR UN CAMPESINO ESPANOL (STOCKCERO),Luis Valdez and El Teatro Campesino
502,spanish civil war 19361939,spanish civil war,9781472810069,9781421366593,THE SPANISH CIVIL WAR : 1936-1939,The Spanish civil war.
503,special educ contemporari societi introduct ex...,sociolog perspect educ children contemporari s...,9781071903100,1799818497,SPECIAL EDUCATION IN CONTEMPORARY SOCIETY : AN...,Sociological perspectives on educating childre...
504,languag disord children,histori chines languag,9780133825879,0429264666,LANGUAGE DISORDERS IN CHILDREN,A history of the Chinese language /


In [48]:
titleComparisonTable.to_excel(date_str+'ISBN Matching Table.xlsx', index = False)

## 8. Continue cleaning up the VolBooks report

Return here once all ISBNs have been fixed above.

In [65]:
combined_df['Portfolio Authentication Note']

0                                            NaN
1                                            NaN
2                               Unlimited Access
3                                            NaN
4                                            NaN
5                                            NaN
6                         Online Access - 1 user
7                                            NaN
8                                            NaN
9                                            NaN
10                                           NaN
11                                           NaN
12                                           NaN
13                                           NaN
14               Online Access - Unlimited users
15               Online Access - Unlimited Users
16                                           NaN
17               Online Access - Unlimited Users
18                       Online Access - 3 users
19                                           NaN
20                  

In [66]:
combined_df['Electronic location split'] = combined_df['Electronic location and access'].apply(lambda x: x.split() if isinstance(x, str) else x)

combined_df['Electronic location'] = combined_df['Electronic location split'].apply(lambda x: ''.join(map(str, x[:1])) if isinstance(x, list) else x)

# Extract items excluding the first item from each list
combined_df['Electronic access'] = combined_df['Electronic location split'].apply(lambda x: ' '.join(x[1:]) if isinstance(x, list) else x)


In [67]:
combined_df['Electronic access']

0                                               
1                                               
2              Unlimited Access Unlimited Access
3                                               
4                                  Click to View
5              Unlimited Access Unlimited Access
6                         Online Access - 1 user
7                                               
8              Unlimited Access Unlimited Access
9                                            NaN
10                                           NaN
11                                              
12                                              
13             Unlimited Access Unlimited Access
14             Unlimited Access Unlimited Access
15                                 Click to View
16           Online access - 1 simultaneous user
17             Unlimited Access Unlimited Access
18                       Single User Single User
19             Unlimited Access Unlimited Access
20                  

## 9. save the alma analytics api output as an excel file

In [68]:
#save the big, concatenated dataframe to disk
combined_df.to_excel(date_str+'_analytics_output.xlsx', index = False)

## do some work on the dataframe

In [69]:
# Create a new column 'Concatenated Notes' with deduplication
combined_df['Concatenated Notes'] = combined_df.apply(concatenate_and_deduplicate_notes, axis=1)

#filter to only show rows where the No. of Available Portfolios is not zero
combined_df = combined_df.loc[combined_df['No. of Available Portfolio'] != 0]


# Display the updated DataFrame
combined_df


,Electronic location and access,ISBN (Normalized),ISBN,MMS Id,Title,Electronic Collection Public Name,Availability,Portfolio Id,Portfolio Public Note,No. of Available Portfolio,Portfolio Authentication Note,Electronic location split,Electronic location,Electronic access,Concatenated Notes
0,https://search.ebscohost.com/login.aspx?direct...,144434594X; 9781444345940; 1444345915; 9781444...,144434594X; 9781444345940; 1444345915; 9781444...,9922520450102311,A companion to paleopathology /,ProQuest Ebook Central DDA Discovery,Available,53366614400002311,Click to View,0,NaN,[https://search.ebscohost.com/login.aspx?direc...,https://search.ebscohost.com/login.aspx?direct...,,Click to View; https://search.ebscohost.com/lo...
1,https://www.taylorfrancis.com/books/9781003183686,1003183689; 9781003183686; 1000811808; 9781000...,1003183689; 9781003183686; 1000811808; 9781000...,9926319485402311,"Deaf people and society : psychological, socio...",Taylor & Francis Ebooks (EBA),Available,53395065450002311,NaN,0,NaN,[https://www.taylorfrancis.com/books/978100318...,https://www.taylorfrancis.com/books/9781003183686,,https://www.taylorfrancis.com/books/9781003183686
2,https://www.degruyter.com/isbn/9781613630174 U...,9781613630174; 1613630174; 9781613630181; 1613...,9781613630174; 1613630174; 9781613630181; 1613...,9926828613202311,Financial Literacy for Managers : Finance and ...,De Gruyter Ebooks Individually Purchased,Available,53446301860002311,NaN,0,Unlimited Access,"[https://www.degruyter.com/isbn/9781613630174,...",https://www.degruyter.com/isbn/9781613630174,Unlimited Access Unlimited Access,Unlimited Access; https://www.degruyter.com/is...
3,https://search.ebscohost.com/login.aspx?direct...,144434594X; 9781444345940; 1444345915; 9781444...,144434594X; 9781444345940; 1444345915; 9781444...,9922520450102311,A companion to paleopathology /,Ebooks moved from ProQuest to EBSCO,Available,53449629450002311,NaN,1,NaN,[https://search.ebscohost.com/login.aspx?direc...,https://search.ebscohost.com/login.aspx?direct...,,https://search.ebscohost.com/login.aspx?direct...
4,https://ebookcentral.proquest.com/lib/utk/deta...,9780470671078; 0470671076; 9781119950264; 1119...,9780470671078; 0470671076; 9781119950264; 1119...,9926276606602311,A guide to old English,ProQuest Ebook Central: Academic Complete,Available,53391009200002311,NaN,1,NaN,[https://ebookcentral.proquest.com/lib/utk/det...,https://ebookcentral.proquest.com/lib/utk/deta...,Click to View,https://ebookcentral.proquest.com/lib/utk/deta...
5,https://muse.jhu.edu/book/46850/ Unlimited Acc...,0809326620; 9780809326624; 9780809326631; 0809...,0809326620; 9780809326624; 9780809326631; 0809...,9926062446102311,Appeals in Modern Rhetoric An Ordinary Languag...,Project Muse Ebooks Individually Purchased,Available,53356492330002311,Unlimited Access,1,NaN,"[https://muse.jhu.edu/book/46850/, Unlimited, ...",https://muse.jhu.edu/book/46850/,Unlimited Access Unlimited Access,Unlimited Access; https://muse.jhu.edu/book/46...
6,http://search.ebscohost.com/login.aspx?direct=...,030738862X; 9780307388629; 9780394535975; 0394...,030738862X; 9780307388629; 9780394535975; 0394...,9925919468902311,Beloved /,EBSCO Ebooks Individually Purchased,Available,53331965650002311,NaN,1,Online Access - 1 user,[http://search.ebscohost.com/login.aspx?direct...,http://search.ebscohost.com/login.aspx?direct=...,Online Access - 1 user,http://search.ebscohost.com/login.aspx?direct=...
7,http://dx.doi.org/10.1007/978-1-4615-0729-1,9781461507291; 1461507294; 9780306472749; 0306...,9781461507291; 1461507294; 9780306472749; 0306...,9925584156402311,Brewing,Springer English/International Ebook Collection,Available,53263558200002311,NaN,1,NaN,[http://dx.doi.org/10.1007/978-1-4615-0729-1],http://dx.doi.org/10.1007/978-1-4615-0729-1,,http://dx.doi.org/10.1007/978-1-4615-0729-1
8,https://www.jstor.org/stable/10.2307/j.ctv2rr3...,194483818X; 9781944838188; 9781944838195; 1944...,194483818X; 9781944838188; 9781944838195; 1944...,99264317

## 10. save the analytics_output file as csv

In [70]:
#save the dataframe to disk once again, only this time as a csv
combined_df.to_csv(date_str+'_analytics_output.csv', index = False)

## 11. Combine Data Files for Exporting

In [71]:
results = pd.read_excel(date_str+'_set_output.xlsx') #From Alma Directly
#results = pd.read_excel('ProQuest to EBSCO Check 2-4.xlsx')
bookstoreList = pd.read_excel('searchresults_'+date_str+'.xlsx') #From VolBooks Directly
textbookISBNs = combined_df #Alma Cleaned-Up Version

## Make sure that any ISBNs that were fixed before loading into Alma are also fixed here!

In [72]:
#Fix known mismatching ISBNs in bookstoreList

for i in range(bookstoreList.shape[0]):
    if bookstoreList.at[i, 'ISBN'] == '9780191037627': #Silas Marner
        print('Found Silas Marner')
        bookstoreList.at[i, 'ISBN'] = '9780191037610'    
    if bookstoreList.at[i, 'ISBN'] == '9781040675892': #Sick Enough
        print('Found Sick Enough')       
        bookstoreList.at[i, 'ISBN'] = '9781041036487'   
    if bookstoreList.at[i, 'ISBN'] == '9781440855375': #Evaluation and Measurement of Library Services
        print('Found Evaluation and Measurement of Library Services')     
        bookstoreList.at[i, 'ISBN'] = '9781440855368'     
    if bookstoreList.at[i, 'ISBN'] == '9781003155119': #Information Science: The Basics
        print('Found Information Science')     
        bookstoreList.at[i, 'ISBN'] = '9780367725204'      
    if bookstoreList.at[i, 'ISBN'] == '9781119628149': #International Financial Statement Analysis
        print('Found International Financial Statement Analysis')    
        bookstoreList.at[i, 'ISBN'] = '9781119682141'     
    if bookstoreList.at[i, 'ISBN'] == '9781351133135': #Counseling Children and Adolescents
        print('Found Counseling Children and Adolescents')  
        bookstoreList.at[i, 'ISBN'] = '9781351133159'   
    if bookstoreList.at[i, 'ISBN'] == '9781501505935': #Demand and Supply Integration
        print('Found Demand and Supply Integration')  
        bookstoreList.at[i, 'ISBN'] = '9781501506024'  
    if bookstoreList.at[i, 'ISBN'] == '9781506352916': #Performance Appraisal and Management
        print('Found Performance Appraisal and Management')  
        bookstoreList.at[i, 'ISBN'] = '9781506352886'  
    if bookstoreList.at[i, 'ISBN'] == '9781317346005': #On the History of Political Philosophy
        print('Found On the History of Political Philosophy')
        bookstoreList.at[i, 'ISBN'] = '9781317346012'
    if bookstoreList.at[i, 'ISBN'] == '9781000690835': #Child-Centered Play Therapy
        print('Found Child-Centered Play Therapy')
        bookstoreList.at[i, 'ISBN'] = '9781003260431' 
    if bookstoreList.at[i, 'ISBN'] == '9781483315430': #Controversies in Contemporary Advertising
        print('Found Controversies in Contemporary Advertising')
        bookstoreList.at[i, 'ISBN'] = '9781452233130' 
    if bookstoreList.at[i, 'ISBN'] == '9781538139332': #Cooperation and Conflict Between State and Local Government
        print('Found Cooperation and Conflict Between State and Local Government')
        bookstoreList.at[i, 'ISBN'] = '9798216405665' 
    if bookstoreList.at[i, 'ISBN'] == '9781315218489': #Advanced Linear Algebra for Engineers with MATLAB
        print('Found Advanced Linear Algebra for Engineers with MATLAB') 
        bookstoreList.at[i, 'ISBN'] = '9781420095241' 
    if bookstoreList.at[i, 'ISBN'] == '9781351834377': #Advanced Linear Algebra for Engineers with MATLAB, second ISBN
        print('Found Advanced Linear Algebra for Engineers with MATLAB')
        bookstoreList.at[i, 'ISBN'] = '9781420095241'  
    if bookstoreList.at[i, 'ISBN'] == '9780393069754': #Beowulf
        print('Found Beowulf') 
        bookstoreList.at[i, 'ISBN'] = '9781350212718'
    if bookstoreList.at[i, 'ISBN'] == '9781483322100': #Conflict, War, and Peace: An Introduction to Scientific Research
        print('Found Conflict, War, and Peace')
        bookstoreList.at[i, 'ISBN'] = '9781071934111'
    if bookstoreList.at[i, 'ISBN'] == '9780742575622': #Criminologiccal Theory: An Analysis of its Underlying Assumptions
        print('Found Criminological Theory')
        bookstoreList.at[i, 'ISBN'] = '9798216404972'
    if bookstoreList.at[i, 'ISBN'] == '9780191007774': #Global Environments Through the Quarternary: Exploring Environmental Change
        print('Found Global Environments Through the Quarternary')
        bookstoreList.at[i, 'ISBN'] = '9780191810169'
    if bookstoreList.at[i, 'ISBN'] == '9780191606373': #History: A Very Short Introduction
        print('Found History')
        bookstoreList.at[i, 'ISBN'] = '9780191540189'
    if bookstoreList.at[i, 'ISBN'] == '9780124079489': #Introduction to Probability Models
        print('Found Introduction to Probability Models')
        bookstoreList.at[i, 'ISBN'] = '9780125984560'
    if bookstoreList.at[i, 'ISBN'] == '9781948057882': #John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition
        print("Found John's Hopkins Evidence-Based Practice for Nurses and Healthcare Professionals, Fourth Edition")
        bookstoreList.at[i, 'ISBN'] = '9781948057899'
    if bookstoreList.at[i, 'ISBN'] == '9780838937952': #Metadata
        print('Found Metadata')
        bookstoreList.at[i, 'ISBN'] = '9780838948637'
    if bookstoreList.at[i, 'ISBN'] == '9780307757890': #Paradise Lost
        print('Found Paradise Lost')
        bookstoreList.at[i, 'ISBN'] = '9781317865735'
    if bookstoreList.at[i, 'ISBN'] == '9781466887787': #14-18: Understanding the Great War
        print('Found 14-18 Understanding the Great War')
        bookstoreList.at[i, 'ISBN'] = '9780809046430'
    if bookstoreList.at[i, 'ISBN'] == '9781400030156': #The Big Sleep
        print('Found The Big Sleep')
        bookstoreList.at[i, 'ISBN'] = '9781479403776'
    if bookstoreList.at[i, 'ISBN'] == '9780191629112': #The Book of Marvels and Travels
        print('Found The Book of Marvels and Travels')
        bookstoreList.at[i, 'ISBN'] = '9780191629105'
    if bookstoreList.at[i, 'ISBN'] == '9780140449617': #The Death of Ivan Ilyich and Other Stories
        print('Found The Death of Ivan Ilyich')
        bookstoreList.at[i, 'ISBN'] = '9780191649042'
    if bookstoreList.at[i, 'ISBN'] == '9781479823550': #Forced Out: Migrant Mothers in Search of Refuge and Hope
        print('Found Forced Out')
        bookstoreList.at[i, 'ISBN'] = '9781479823529'
    if bookstoreList.at[i, 'ISBN'] == '9780198745686': #Imperial Apocalypse: The Great War and the Destruction of the Russian Empire
        print('Found Imperial Apocalypse')
        bookstoreList.at[i, 'ISBN'] = '9781644699522'
    if bookstoreList.at[i, 'ISBN'] == '9780262367509': #Introduction to Algorithms, Fourth Edition
        print('Found Introduction to Algorithms')
        bookstoreList.at[i, 'ISBN'] = '9780262046305'
    if bookstoreList.at[i, 'ISBN'] == '9780813178288': #John Hervey Wheeler, Black Banking, and the Economic Struggle for Civil Rights]
        print('Found John Hervey Wheeler')
        bookstoreList.at[i, 'ISBN'] = '9780813178271'
    if bookstoreList.at[i, 'ISBN'] == '9781452289236': #Mental Health Case Management: A Practical Guide
        print('Found Mental Health Case Management')
        bookstoreList.at[i, 'ISBN'] = '9781452256078'
    if bookstoreList.at[i, 'ISBN'] == '9781101576984': #Merchants of Culture The Publishing Business in the Twenty-First Century
        print('Found Merchants of Culture')
        bookstoreList.at[i, 'ISBN'] = '9780745661421'
    if bookstoreList.at[i, 'ISBN'] == '9781319049966': #Narrative of the Life of Frederick Douglass: An American Slave, Written by Himself
        print('Found Frederick Douglass')
        bookstoreList.at[i, 'ISBN'] = '9780760355466'
    if bookstoreList.at[i, 'ISBN'] == '9780199720149': #The Oxford Handbook of Engineering and Technology in the Classical World
        print('Found Handbook of Engineering and Technology')
        bookstoreList.at[i, 'ISBN'] = '9780195187311'
    if bookstoreList.at[i, 'ISBN'] == '9781350113008': #Performance Drawing
        print('Found Performance Drawing')
        bookstoreList.at[i, 'ISBN'] = '9781350113015'
    if bookstoreList.at[i, 'ISBN'] == '9780191624711': #Persuasion
        print('Found Persuasion')
        bookstoreList.at[i, 'ISBN'] = '9781316848630'
    if bookstoreList.at[i, 'ISBN'] == '9781009314848': #The Gas Mask in Interwar Germany: Visions of Chemical Modernity
        print('Found Gas Mask')
        bookstoreList.at[i, 'ISBN'] = '9781009314862'
    if bookstoreList.at[i, 'ISBN'] == '9780385093309': #Good-Bye to All That
        print('Found Good-Bye to All That')
        bookstoreList.at[i, 'ISBN'] = '9781774649367'
    if bookstoreList.at[i, 'ISBN'] == '9781119837909': #Research Ethics for Scientists
        print('Found Research Ethics for Scientists')
        bookstoreList.at[i, 'ISBN'] = '9781119978862'
    if bookstoreList.at[i, 'ISBN'] == '9780307432568': #The Return of the Soldier
        print('Found The Return of the Soldier')
        bookstoreList.at[i, 'ISBN'] = '9780486846545'
    if bookstoreList.at[i, 'ISBN'] == '9781496203168': #Siberian Exile
        print('Found Siberian Exile')
        bookstoreList.at[i, 'ISBN'] = '9780803299597'
    if bookstoreList.at[i, 'ISBN'] == '9780190879976': #Teaching Music Theory
        print('Found Teaching Music Theory')
        bookstoreList.at[i, 'ISBN'] = '9780197510575'
    if bookstoreList.at[i, 'ISBN'] == '9780197644058': #An Uneasy Peace
        print('Found An Uneasy Peace')
        bookstoreList.at[i, 'ISBN'] = '9780197619407'
    if bookstoreList.at[i, 'ISBN'] == '9781975121549': #Fleisher & Ludwig's Textbook of Pediatric Emergency Medicine
        print('Found Fleisher & Ludwig Textbook')
        bookstoreList.at[i, 'ISBN'] = '9781975121532'
    if bookstoreList.at[i, 'ISBN'] == '9781482542222': #Treating Complex Traumatic Stress Disorders in Adults: Scientific Foundations and Therapeutic Models
        print('Found Treating Complex Traumatic Stress Disorders in Adults')
        bookstoreList.at[i, 'ISBN'] = '9781462542239'
    if bookstoreList.at[i, 'ISBN'] == '9781316044582': #Elements of Mathematical Ecology
        print('Found Elements of Mathematical Ecology')
        bookstoreList.at[i, 'ISBN'] = '9780511608520'
    if bookstoreList.at[i, 'ISBN'] == '9780307758071': #Herland
        print('Found Herland')
        bookstoreList.at[i, 'ISBN'] = '9781667601038'
    if bookstoreList.at[i, 'ISBN'] == '9780393270372': #Narrative of the Life of Frederick Douglass
        print('Found Frederick Douglass')
        bookstoreList.at[i, 'ISBN'] = '9780760355466'
    if bookstoreList.at[i, 'ISBN'] == '9780199748037': #Population Health: Concepts and Methods
        print('Found Population Health')
        bookstoreList.at[i, 'ISBN'] = '9780199864379'
    if bookstoreList.at[i, 'ISBN'] == '9780300048124': #Preschool in Three Cultures: Japan, China and the United States
        print('Found Preschool in Three Cultures')
        bookstoreList.at[i, 'ISBN'] = '9780585358543'
    if bookstoreList.at[i, 'ISBN'] == '9781101078143': #The Souls of BLack Folk
        print('Found The Souls of Black Folk')
        bookstoreList.at[i, 'ISBN'] = '9781315631998'
    if bookstoreList.at[i, 'ISBN'] == '0679601813': #The Brothers Karamazov
        print('Found The Brothers Karamazov')
        bookstoreList.at[i, 'ISBN'] = '9781786565556'
    if bookstoreList.at[i, 'ISBN'] == '9781612495736': #Manufacturing Facilities Design and Material Handling: Sixth Edition
        print('Found Manufacturing Facilities Design')
        bookstoreList.at[i, 'ISBN'] = '9781612495729'
    if bookstoreList.at[i, 'ISBN'] == '9780191605475': #Bleak House
        print('Found Bleak House')
        bookstoreList.at[i, 'ISBN'] = '9780486822556'
    if bookstoreList.at[i, 'ISBN'] == '9781119348450': #Modern Nuclear Chemistry
        print('Found Modern Nuclear Chemistry')
        bookstoreList.at[i, 'ISBN'] = '9780471768623'
    if bookstoreList.at[i, 'ISBN'] == '9781903544877': #Planetary Geology: An Introduction
        print('Found Planetary Geology')
        bookstoreList.at[i, 'ISBN'] = '9781780461045'
    if bookstoreList.at[i, 'ISBN'] == '9783031758539': #R for Political Science Research: An Introduction for Absolute Beginners
        print('Found R for Political Science Research')
        bookstoreList.at[i, 'ISBN'] = '9783031758522'
    if bookstoreList.at[i, 'ISBN'] == '9780674040793': #Clinging to Mammy: The Faithful Slave in Twentieth-Century America
        print('Found Clinging to Mammy')
        bookstoreList.at[i, 'ISBN'] = '9780674024335'
    if bookstoreList.at[i, 'ISBN'] == '9781460404027': #The Return of the Soldier
        print('Found Return of the Soldier')
        bookstoreList.at[i, 'ISBN'] = '9780486846545'
    if bookstoreList.at[i, 'ISBN'] == '9781315365619': #Instrumentation, Measurements, and Experiments in Fluids
        print('Found Experiments in Fluids')
        bookstoreList.at[i, 'ISBN'] = '9780849307591'
    if bookstoreList.at[i, 'ISBN'] == '9781612930428': #Pride and Prejudice
        print('Found Pride and Prejudice')
        bookstoreList.at[i, 'ISBN'] = '9781504033831'
    if bookstoreList.at[i, 'ISBN'] == '9781317615880': #Environmental Ethics for the Long Term
        print('Found Environmental Ethics for the Long Term')
        bookstoreList.at[i, 'ISBN'] = '9781317615897'
    if bookstoreList.at[i, 'ISBN'] == '9781119943501': #The Eyes of the Skin
        print('Found The Eyes of the Skin')
        bookstoreList.at[i, 'ISBN'] = '9781394200702'
    if bookstoreList.at[i, 'ISBN'] == '9780806130576': #Horace Epodes and Odes
        print('Found Horace Epodes and Odes')
        bookstoreList.at[i, 'ISBN'] = '0585165475'
    if bookstoreList.at[i, 'ISBN'] == '9781119585473': #Nonprofit Management 101
        print('Found Nonprofit Management 101')
        bookstoreList.at[i, 'ISBN'] = '9781119585527'
    if bookstoreList.at[i, 'ISBN'] == '9780062466679': #A People's History of the United States
        print("Found A People's History of the United States")
        bookstoreList.at[i, 'ISBN'] = '9781317325291'

Found Experiments in Fluids
Found Experiments in Fluids
Found Frederick Douglass
Found Horace Epodes and Odes
Found Pride and Prejudice
Found Herland
Found Herland
Found Frederick Douglass
Found The Eyes of the Skin
Found Preschool in Three Cultures
Found Preschool in Three Cultures
Found Environmental Ethics for the Long Term
Found Environmental Ethics for the Long Term
Found On the History of Political Philosophy
Found On the History of Political Philosophy
Found On the History of Political Philosophy
Found Population Health
Found A People's History of the United States
Found Nonprofit Management 101
Found Nonprofit Management 101
Found Herland
Found Herland


In [73]:
#pd.set_option('display.max_rows', None)
results.sort_values(by = 'Title')['Title'].unique()

<ArrowStringArray>
[                                                                                                       'A Pocket Style Manual [electronic resource].',
                                                                                           'A companion to paleopathology / edited by Anne L. Grauer.',
                                                                                          'A guide to Old English / Bruce Mitchell, Fred C. Robinson.',
                                                                 'A guide to old English [electronic resource] / Bruce Mitchell and Fred C. Robinson.',
                                                             'A people's history of the United States / Howard Zinn ; introduction by Anthony Arnove.',
                  'A practical guide to action research and teacher enquiry : making a difference in the early years / Amanda Ince and Eleanor Kitto.',
                                              'After the New Testamen

In [74]:
textbookISBNs = textbookISBNs.astype({'MMS Id': 'int64'})



resultsPlusList = pd.merge(results, textbookISBNs, left_on = 'MMS ID', right_on = 'MMS Id', how = 'inner')

In [75]:
resultsPlusList

,Title_x,Alternate Graphic Representation,Type,Creator / Imprint,Subject,Series,Relation,Modification Date,Edition,Uniform Title,Medium Type,Language,Creation Date,Language of Cataloging,ISBN_x,ISBN (13),ISSN,Cluster ISSN,Record Number,MMS ID,Bibliographic Rank,Local Call Number,Other Classification Number,NLM Call Number,Dewey Decimal Classification Number,LC Call Number,Classification Part,UDC Classification Number,NAL Call Number,Handle Identifier,Peer Reviewed,Open Access,Physical Availability,Digital Availability,Electronic Availability,Collection Name,Collection ID,Orders,Requests,Physical Description Extent,Local Labels,Related Records,Courses,Licenses,Reminders,Notes,Short Title,Electronic location and access,ISBN (Normalized),ISBN_y,MMS Id,Title_y,Electronic Collection Public Name,Availability,Portfolio Id,Portfolio Public Note,No. of Available Portfolio,Portfolio Authentication Note,Electronic location split,Electronic location,Electronic access,Concatenated Notes
0,The anxious generation : how the great rewirin...,NaN,Book {Book - Electronic} still image; computer...,"Haidt, Jonathan, (New York : Penguin Press, [2...",Children--United States--Social conditions--21...,NaN,NaN,2026-10-30 06:41:00,NaN,NaN,NaN,English,2024-04-17 06:03:00,English,9780593655047; 0593655044; 9781802063288; 1802...,9780593655047; 9780593655047; 9781802063288; 9...,NaN,NaN,(OCoLC)on1416597966; (OCoLC)1416597966; (OCoLC...,9926536272302311,101,NaN,NaN,NaN,NaN,HQ792.U5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at ProQuest; ProQuest Ebook...,NaN,NaN,1,0,1 online resource (385 pages) :,NaN,1,0,1,0,0,The anxious generation :,https://search.ebscohost.com/login.aspx?direct...,0593655044; 9780593655047; 1802063285; 9781802...,0593655044; 9780593655047; 1802063285; 9781802...,9926536272302311,The anxious generation : how the great rewirin...,Ebooks moved from ProQuest to EBSCO,Available,53450796290002311,NaN,1,Online Access - 1 user,[https://search.ebscohost.com/login.aspx?direc...,https://search.ebscohost.com/login.aspx?direct...,,https://search.ebscohost.com/login.aspx?direct...
1,Appeals in Modern Rhetoric An Ordinary Languag...,NaN,Book {Book - Electronic} text; computer; onlin...,"Killingsworth, M. Jimmie. (Carbondale, Ill. : ...",Report writing.;Persuasion (Rhetoric);English ...,Book collections on Project MUSE.,NaN,2022-01-25 04:12:00,NaN,NaN,NaN,English,2021-08-12 12:33:00,NaN,0809326620 (alk. paper); 0809326639 (pbk. : al...,9780809326624 (alk. paper); 9780809326631 (pbk...,NaN,NaN,(CaONFJC)cis035298712,9926062446102311,88,NaN,NaN,NaN,808/.042,PE1431 .K55 2005,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at Project Muse; Project Mu...,NaN,NaN,1,0,"1 online resource (xiii, 172 p.)",NaN,0,0,0,0,0,Appeals in Modern Rhetoric,https://muse.jhu.edu/book/46850/ Unlimited Acc...,0809326620; 9780809326624; 9780809326631; 0809...,0809326620; 9780809326624; 9780809326631; 0809...,9926062446102311,Appeals in Modern Rhetoric An Ordinary Languag...,Project Muse Ebooks Individually Purchased,Available,53356492330002311,Unlimited Access,1,NaN,"[https://muse.jhu.edu/book/46850/, Unlimited, ...",https://muse.jhu.edu/book/46850/,Unlimited Access Unlimited Access,Unlimited Access; https://muse.jhu.edu/book/46...
2,The art of the pitch [electronic resource] : p...,NaN,Book {Book - Electronic} text; computer; onlin...,"Coughter, Peter. (New York : Palgrave Macmilla...",Business presentations.;Persuasion (Psychology...,NaN,NaN,2026-11-05 04:13:00,1st ed.,NaN,NaN,English,2025-05-28 12:33:00,NaN,9780230120518 (hardcover); 9781137512338 (Spri...,9780230120518 (hardcover); 9781137512338 (Spri...,NaN,NaN,(CaONFJC)cis036311057,9926737922302311,87,NaN,NaN,NaN,658.4/52,HF5718.22 .C68 2012eb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at Springer Nature; Springe...,NaN,NaN,1,0,"1 online resource (x, 240 p.) :",NaN,1,0,0,0,0,The art of the pitch,https://link.springer.com/978-1-137-51233-8 Un...,9780230120518; 0230120512; 9781137512338; 1137...,978023012

In [76]:
#resultsPlusList.to_excel('Results Plus List Example.xlsx')

resultsPlusList.sort_values(by = 'Title_x')['Title_x'].unique()

<ArrowStringArray>
[                                                                                                                                                                                                                                                                              'A Pocket Style Manual [electronic resource].',
                                                                                                                                                                                                                                                                  'A companion to paleopathology / edited by Anne L. Grauer.',
                                                                                                                                                                                                                                        'A guide to old English [electronic resource] / Bruce Mitchell and Fred C. Robinson.',
                        

### Expand the ISBN Normalized Column

In [77]:
resultsPlusList['ISBN (Normalized) List'] = resultsPlusList['ISBN (Normalized)'].apply(lambda x: x.split(";"))

In [78]:
expandedList = resultsPlusList.explode('ISBN (Normalized) List')

expandedList['ISBN (Normalized) List'] = expandedList['ISBN (Normalized) List'].str.strip()

#expandedList.to_excel('Expanded List Example.xlsx')

In [79]:
finalJoin = pd.merge(expandedList, bookstoreList, left_on = 'ISBN (Normalized) List', right_on = 'ISBN', how = 'inner')

finalJoin = finalJoin.rename(columns = {'Short Title_y': 'Short Title'})

finalJoin

,Title_x,Alternate Graphic Representation,Type,Creator / Imprint,Subject,Series,Relation,Modification Date,Edition,Uniform Title,Medium Type,Language,Creation Date,Language of Cataloging,ISBN_x,ISBN (13),ISSN,Cluster ISSN,Record Number,MMS ID,Bibliographic Rank,Local Call Number,Other Classification Number,NLM Call Number,Dewey Decimal Classification Number,LC Call Number,Classification Part,UDC Classification Number,NAL Call Number,Handle Identifier,Peer Reviewed,Open Access,Physical Availability,Digital Availability,Electronic Availability,Collection Name,Collection ID,Orders,Requests,Physical Description Extent,Local Labels,Related Records,Courses,Licenses,Reminders,Notes,Short Title_x,Electronic location and access,ISBN (Normalized),ISBN_y,MMS Id,Title_y,Electronic Collection Public Name,Availability,Portfolio Id,Portfolio Public Note,No. of Available Portfolio,Portfolio Authentication Note,Electronic location split,Electronic location,Electronic access,Concatenated Notes,ISBN (Normalized) List,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity
0,The anxious generation : how the great rewirin...,NaN,Book {Book - Electronic} still image; computer...,"Haidt, Jonathan, (New York : Penguin Press, [2...",Children--United States--Social conditions--21...,NaN,NaN,2026-10-30 06:41:00,NaN,NaN,NaN,English,2024-04-17 06:03:00,English,9780593655047; 0593655044; 9781802063288; 1802...,9780593655047; 9780593655047; 9781802063288; 9...,NaN,NaN,(OCoLC)on1416597966; (OCoLC)1416597966; (OCoLC...,9926536272302311,101,NaN,NaN,NaN,NaN,HQ792.U5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at ProQuest; ProQuest Ebook...,NaN,NaN,1,0,1 online resource (385 pages) :,NaN,1,0,1,0,0,The anxious generation :,https://search.ebscohost.com/login.aspx?direct...,0593655044; 9780593655047; 1802063285; 9781802...,0593655044; 9780593655047; 1802063285; 9781802...,9926536272302311,The anxious generation : how the great rewirin...,Ebooks moved from ProQuest to EBSCO,Available,53450796290002311,NaN,1,Online Access - 1 user,[https://search.ebscohost.com/login.aspx?direc...,https://search.ebscohost.com/login.aspx?direct...,,https://search.ebscohost.com/login.aspx?direct...,9780593655047,SPRING 2027,CINEMA STUDIES,202720-CNST.312.001-23211,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,10
1,The anxious generation : how the great rewirin...,NaN,Book {Book - Electronic} still image; computer...,"Haidt, Jonathan, (New York : Penguin Press, [2...",Children--United States--Social conditions--21...,NaN,NaN,2026-10-30 06:41:00,NaN,NaN,NaN,English,2024-04-17 06:03:00,English,9780593655047; 0593655044; 9781802063288; 1802...,9780593655047; 9780593655047; 9781802063288; 9...,NaN,NaN,(OCoLC)on1416597966; (OCoLC)1416597966; (OCoLC...,9926536272302311,101,NaN,NaN,NaN,NaN,HQ792.U5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Electronic version at ProQuest; ProQuest Ebook...,NaN,NaN,1,0,1 online resource (385 pages) :,NaN,1,0,1,0,0,The anxious generation :,https://search.ebscohost.com/login.aspx?direct...,0593655044; 9780593655047; 1802063285; 9781802...,0593655044; 9780593655047; 1802063285; 9781802...,9926536272302311,The anxious generation : how the great rewirin...,Ebooks moved from ProQuest to EBSCO,Available,53450796290002311,NaN,1,Online Access - 1 user,[https://search.ebscohost.com/login.aspx?direc...,https://search.ebscohost.com/login.aspx?direct...,,https://search.ebscohost.com/login.aspx?direct...,9780593655047,SPRING 2027,CINEMA STUDIES,202720-CNST.312.005-41752,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,5
2,The anxious generation : how the great rewirin...,NaN,Book {Book - Electronic} still image; computer...,"Haidt, Jonathan, (New York : Penguin Press, [2...",Children--United States--Social conditions--21...,NaN,NaN,2026-10-30 06:41:00,NaN,NaN,NaN,English,2024-04-17 06:03:00,English,9780593

In [80]:
selectiveCols = finalJoin[['Academic Term', 'Academic Department Name', 'Section Code', 'Instructor Name', 'ISBN (Normalized) List', 'Short Title', 'Book Author', 'Book Publisher', 'Enrollment Capacity', 'MMS ID', 'Electronic location', 'Electronic access', 'Electronic Availability']]

selectiveCols = selectiveCols.drop_duplicates().reset_index()

selectiveCols = selectiveCols.rename(columns = {'Section Code': 'Course Code'})

len(selectiveCols.sort_values(by = 'Short Title')['Short Title'].unique())

70

In [81]:
selectiveCols.sort_values(by = 'Short Title')['Short Title'].unique()

<ArrowStringArray>
[                                                                      'A COMPANION TO PALEOPATHOLOGY',
                                                             'A PEOPLE'S HISTORY OF THE UNITED STATES',
                                                                               'A POCKET STYLE MANUAL',
   'A PRACTICAL GUIDE TO ACTION RESEARCH AND TEACHER ENQUIRY : MAKING A DIFFERENCE IN THE EARLY YEARS',
                                          'APPEALS IN MODERN RHETORIC : AN ORDINARY LANGUAGE APPROACH',
                                                           'ARTIFICIAL INTELLIGENCE FOR CYBERSECURITY',
                                                                                             'BELOVED',
                                                                                             'BREWING',
              'BUSINESS MODEL GENERATION : A HANDBOOK FOR VISIONARIES, GAME CHANGERS, AND CHALLENGERS',
                                'CASE STUDIES

In [82]:
len(selectiveCols.sort_values(by = 'Short Title')['Short Title'].unique())

70

In [83]:
nonUnlimited = finalJoin[~finalJoin['Short Title'].isin(selectiveCols['Short Title'])].dropna(how = 'all')

nonUnlimited.sort_values(by = 'Short Title')['Short Title'].unique()

<ArrowStringArray>
[]
Length: 0, dtype: str

## 12. Use the code block below to edit information in the VolBooks file as needed

This should be done when a title has multiple holdings in our collections, but only one fits the criteria for VolBooks submission. Coding it here will save manual editing time later.

In [84]:
warnings.filterwarnings("ignore")

bookstoreExport = selectiveCols

# Fill in the Primo Link based on the MMS ID

for i in range(len(bookstoreExport['MMS ID'])):
    bookstoreExport.iloc[i, bookstoreExport.columns.get_loc('Electronic location')] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma' + str(bookstoreExport.iloc[i, bookstoreExport.columns.get_loc('MMS ID')])

bookstoreExport = bookstoreExport.rename(columns = {'ISBN (Normalized) List': 'ISBN', 'Electronic access': 'Access_Type', 'Electronic location': 'Permalink', 'Course Code': 'Section Code'})

bookstoreExport = bookstoreExport[['Academic Term', 'Academic Department Name', 'Section Code', 'Instructor Name', 'ISBN', 'Short Title', 'Book Author', 'Book Publisher', 'Enrollment Capacity', 'MMS ID', 'Access_Type', 'Electronic Availability', 'Permalink']]

bookstoreExport['Section Title Code'] = bookstoreExport['Section Code'] + bookstoreExport['Short Title']

bookstoreExport

,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity,MMS ID,Access_Type,Electronic Availability,Permalink,Section Title Code
0,SPRING 2027,CINEMA STUDIES,202720-CNST.312.001-23211,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,10,9926536272302311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-CNST.312.001-23211THE ANXIOUS GENERATION
1,SPRING 2027,CINEMA STUDIES,202720-CNST.312.005-41752,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,5,9926536272302311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-CNST.312.005-41752THE ANXIOUS GENERATION
2,SPRING 2027,POLITICAL SCIENCE,202720-POLS.312.001-20474,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,40,9926536272302311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-POLS.312.001-20474THE ANXIOUS GENERATION
3,SPRING 2027,POLITICAL SCIENCE,202720-POLS.312.004-34987,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,5,9926536272302311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-POLS.312.004-34987THE ANXIOUS GENERATION
4,SPRING 2027,ENGLISH,202720-ENGL.355.003-25830,"Saenkhum, Tanita",9780809326631,APPEALS IN MODERN RHETORIC : AN ORDINARY LANGU...,"KILLINGSWORTH, M. JIMMIE",SOUTHERN ILLINOIS UNIVERSITY PRESS,25,9926062446102311,Unlimited Access Unlimited Access,Electronic version at Project Muse; Project Mu...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ENGL.355.003-25830APPEALS IN MODERN RHE...
5,SPRING 2027,ADVERTISING,202720-ADVT.450.001-20034,"Thomas, Dustin",9781137512338,THE ART OF THE PITCH,"COUGHTER, PETER",PALGRAVE MACMILLAN,40,9926737922302311,Unlimited Access Unlimited Access,Electronic version at Springer Nature; Springe...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ADVT.450.001-20034THE ART OF THE PITCH
6,SPRING 2027,ADVERTISING,202720-ADVT.470.003-36234,"Thomas, Dustin",9781137512338,THE ART OF THE PITCH,"COUGHTER, PETER",PALGRAVE MACMILLAN,10,9926737922302311,Unlimited Access Unlimited Access,Electronic version at Springer Nature; Springe...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ADVT.470.003-36234THE ART OF THE PITCH
7,SPRING 2027,ENGLISH,202720-ENGL.253.002-20648,"Newburn, Harry",9780307388629,BELOVED,"MORRISON, TONI",KNOPF DOUBLEDAY PUBLISHING GROUP,25,9925919468902311,Online Access - 1 user,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ENGL.253.002-20648BELOVED
8,SPRING 2027,FOOD SCIENCE,202720-FDSC.210.001-37487,"Sommer, Stephan",9780306472749,BREWING,"YOUNG, TOM W. AND LEWIS, MICHAEL J.",SPRINGER,45,9925584156402311,,Electronic version at Springer Nature; Springe...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-FDSC.210.001-37487BREWING
9,SPRING 2027,EDUCATION OF THE DEAF AND HARD OF HEARING,202720-EDDE.425.004-36357,"Skyer, Michael",9781944838195,"CASE STUDIES IN DEAF EDUCATION : INQUIRY, APPL...","GUARDINO, CAROLINE, BEAL, JENNIFER S., CANNON,...",GALLAUDET UNIVERSITY PRESS,25,9926431786402311,Unlimited Access Unlimited Access,Electronic version at JSTOR; JSTOR Ebooks (JST...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-EDDE.425.004-36357CASE STUDIES IN DEAF ...


In [85]:
for i in range(bookstoreExport.shape[0]):
    if bookstoreExport.at[i, 'Short Title'] == '58-IN-MIND : MULTILINGUAL TEACHING STRATEGIES FOR DIVERSE DEAF STUDENTS':
        bookstoreExport.at[i, 'MMS ID'] = '9926829233602311'
        bookstoreExport.at[i, 'Access_Type'] = 'Unlimited Access Unlimited Access'
        bookstoreExport.at[i, 'Electronic Availability'] = 'Electronic version at JSTOR; JSTOR Ebooks (JSTOR Rialto Implementation); Full Text; Available; 53441315590002311'
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926829233602311'
  
    if bookstoreExport.at[i, 'Short Title'] == 'SKILL ACQUISITION IN SPORT : THEORY AND PRACTICE':
        bookstoreExport.at[i, 'MMS ID'] = '9925871824802311'
        bookstoreExport.at[i, 'Access_Type'] = 'Online Access - Unlimited users (DRM-free)'
        bookstoreExport.at[i, 'Electronic Availability'] = 'Electronic version at Taylor & Francis; Taylor & Francis Ebooks (Taylor & Francis Ebooks Individually Purchased); Full Text; Available; 53323272990002311'
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9925871824802311'
 
    if bookstoreExport.at[i, 'Short Title'] == 'THE COMPREHENSIVE GUIDE TO INTERDISCIPLINARY VETERINARY SOCIAL WORK':
        bookstoreExport.at[i, 'MMS ID'] = '9926829231602311'
        bookstoreExport.at[i, 'Access_Type'] = 'Unlimited Access Unlimited Access'
        bookstoreExport.at[i, 'Electronic Availability'] = 'Electronic version at Springer Nature; Springer Ebooks (Springer Ebooks Individually Purchased); Full Text; Available; 53433284910002311'
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926829231602311'

    if bookstoreExport.at[i, 'Short Title'] == 'CHIEF SUSTAINABILITY OFFICERS AT WORK':
        bookstoreExport.at[i, 'MMS ID'] = '9926909978602311'
        bookstoreExport.at[i, 'Access_Type'] = ''
        bookstoreExport.at[i, 'Electronic Availability'] = "Electronic version at ProQuest; O'Reilly for Higher Education (O'Reilly Online Learning: Academic/Public Library Edition); Full Text; Available; 53452720100002311"
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926909978602311'

    if bookstoreExport.at[i, 'Short Title'] == 'THE MUSEUM OF UNNATURAL HISTORIES':
        bookstoreExport.at[i, 'MMS ID'] = '9926839378502311'
        bookstoreExport.at[i, 'Access_Type'] = 'Full text available:'
        bookstoreExport.at[i, 'Electronic Availability'] = 'Electronic version at Project Muse; Project Muse Ebooks (Project Muse Ebooks Individually Purchased); Full Text; Available; 53446391740002311'
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926839378502311'

    if bookstoreExport.at[i, 'Short Title'] == 'DIFFERENTIAL EQUATIONS, DYNAMIC SYS & INTRO TO CHAOS':
        bookstoreExport.at[i, 'MMS ID'] = '9925873484102311'
        bookstoreExport.at[i, 'Access_Type'] = 'Click to View'
        bookstoreExport.at[i, 'Electronic Availability'] = 'Electronic version at ProQuest; ProQuest Ebook Central: Academic Complete; Full Text; Available; 53323574290002311'
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9925873484102311'

    if bookstoreExport.at[i, 'Short Title'] == 'INFORMATION ARCHITECTURE FOR THE WORLD WIDE WEB':
        bookstoreExport.at[i, 'MMS ID'] = '9926909740702311'
        bookstoreExport.at[i, 'Access_Type'] = ''
        bookstoreExport.at[i, 'Electronic Availability'] = "Electronic version at ProQuest; O'Reilly for Higher Education (O'Reilly Online Learning: Academic/Public Library Edition); Full Text; Available; 53452757920002311"
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926909740702311'

    if bookstoreExport.at[i, 'Short Title'] == 'MOVEMENTS IN ORGANIZATIONAL COMMUNICATION RESEARCH':
        bookstoreExport.at[i, 'MMS ID'] = '9926414519802311'
        bookstoreExport.at[i, 'Access_Type'] = 'Online Access- Unlimited users (DRM-Free)'
        bookstoreExport.at[i, 'Electronic Availability'] = 'Electronic version at Taylor & Francis; Taylor & Francis Ebooks (Taylor & Francis Ebooks EBA Purchased); Full Text; Available; 53407938040002311'
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926414519802311'

    if bookstoreExport.at[i, 'Short Title'] == 'MOVEMENTS IN ORGANIZATIONAL COMMUNICATION RESEARCH : CURRENT ISSUES AND FUTURE DIRECTIONS':
        bookstoreExport.at[i, 'MMS ID'] = '9926414519802311'
        bookstoreExport.at[i, 'Access_Type'] = 'Online Access- Unlimited users (DRM-Free)'
        bookstoreExport.at[i, 'Electronic Availability'] = 'Electronic version at Taylor & Francis; Taylor & Francis Ebooks (Taylor & Francis Ebooks EBA Purchased); Full Text; Available; 53407938040002311'
        bookstoreExport.at[i, 'Permalink'] = 'https://utk.primo.exlibrisgroup.com/permalink/01UTN_KNOXVILLE/bcmt7h/alma9926414519802311'

bookstoreExport = bookstoreExport.drop_duplicates().reset_index()

bookstoreExport

,index,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity,MMS ID,Access_Type,Electronic Availability,Permalink,Section Title Code
0,0,SPRING 2027,CINEMA STUDIES,202720-CNST.312.001-23211,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,10,9926536272302311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-CNST.312.001-23211THE ANXIOUS GENERATION
1,1,SPRING 2027,CINEMA STUDIES,202720-CNST.312.005-41752,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,5,9926536272302311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-CNST.312.005-41752THE ANXIOUS GENERATION
2,2,SPRING 2027,POLITICAL SCIENCE,202720-POLS.312.001-20474,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,40,9926536272302311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-POLS.312.001-20474THE ANXIOUS GENERATION
3,3,SPRING 2027,POLITICAL SCIENCE,202720-POLS.312.004-34987,"Nownes, Anthony",9780593655047,THE ANXIOUS GENERATION,"HAIDT, JONATHAN",PENGUIN PUBLISHING GROUP,5,9926536272302311,,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-POLS.312.004-34987THE ANXIOUS GENERATION
4,4,SPRING 2027,ENGLISH,202720-ENGL.355.003-25830,"Saenkhum, Tanita",9780809326631,APPEALS IN MODERN RHETORIC : AN ORDINARY LANGU...,"KILLINGSWORTH, M. JIMMIE",SOUTHERN ILLINOIS UNIVERSITY PRESS,25,9926062446102311,Unlimited Access Unlimited Access,Electronic version at Project Muse; Project Mu...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ENGL.355.003-25830APPEALS IN MODERN RHE...
5,5,SPRING 2027,ADVERTISING,202720-ADVT.450.001-20034,"Thomas, Dustin",9781137512338,THE ART OF THE PITCH,"COUGHTER, PETER",PALGRAVE MACMILLAN,40,9926737922302311,Unlimited Access Unlimited Access,Electronic version at Springer Nature; Springe...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ADVT.450.001-20034THE ART OF THE PITCH
6,6,SPRING 2027,ADVERTISING,202720-ADVT.470.003-36234,"Thomas, Dustin",9781137512338,THE ART OF THE PITCH,"COUGHTER, PETER",PALGRAVE MACMILLAN,10,9926737922302311,Unlimited Access Unlimited Access,Electronic version at Springer Nature; Springe...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ADVT.470.003-36234THE ART OF THE PITCH
7,7,SPRING 2027,ENGLISH,202720-ENGL.253.002-20648,"Newburn, Harry",9780307388629,BELOVED,"MORRISON, TONI",KNOPF DOUBLEDAY PUBLISHING GROUP,25,9925919468902311,Online Access - 1 user,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ENGL.253.002-20648BELOVED
8,8,SPRING 2027,FOOD SCIENCE,202720-FDSC.210.001-37487,"Sommer, Stephan",9780306472749,BREWING,"YOUNG, TOM W. AND LEWIS, MICHAEL J.",SPRINGER,45,9925584156402311,,Electronic version at Springer Nature; Springe...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-FDSC.210.001-37487BREWING
9,9,SPRING 2027,EDUCATION OF THE DEAF AND HARD OF HEARING,202720-EDDE.425.004-36357,"Skyer, Michael",9781944838195,"CASE STUDIES IN DEAF EDUCATION : INQUIRY, APPL...","GUARDINO, CAROLINE, BEAL, JENNIFER S., CANNON,...",GALLAUDET UNIVERSITY PRESS,25,9926431786402311,Unlimited Access Unlimited Access,Electronic version at JSTOR; JSTOR Ebooks (JST...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-EDDE.425.004-36357CASE STUDIES IN DEAF ...


In [86]:
bookstoreExport.to_excel('Bookstore Access Check ' + date_str + '.xlsx')

# 13. Check Bookstore Access

Use this table to make sure all of the access labels from Alma are correct.

In [87]:
# If this is the first list of the semester, do not run this cell
alreadyChecked1 = pd.read_excel(r'Bookstore Access Check 2026-11-20.xlsx')

alreadyChecked = pd.concat([alreadyChecked1])

alreadyChecked['Section Title Code'] = alreadyChecked['Section Code'] + alreadyChecked['Short Title']

alreadyChecked

,Access Check,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity,MMS ID,Access_Type,Electronic Availability,Permalink,Section Title Code
0,NO GOOD - 3 COPIES,SPRING 2027,ANTHROLOPOGY,202720-ANTH.472.001-32876,"Williams, Amanda",9781444345926,A COMPANION TO PALEOPATHOLOGY,GRAUER,"WILEY & SONS, INCORPORATED, JOHN",12,9922520450102312,NaN,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ANTH.472.001-32876A COMPANION TO PALEOP...
1,NO GOOD - 3 COPIES,SPRING 2027,ANTHROLOPOGY,202720-ANTH.472.002-38419,"Williams, Amanda",9781444345926,A COMPANION TO PALEOPATHOLOGY,GRAUER,"WILEY & SONS, INCORPORATED, JOHN",12,9922520450102312,NaN,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ANTH.472.002-38419A COMPANION TO PALEOP...
2,GOOD,SPRING 2027,SOCIOLOGY,202720-SOCI.310.002-33903,"Lowery, Grady",9781317325291,A PEOPLE'S HISTORY OF THE UNITED STATES,"ZINN, HOWARD",HARPERCOLLINS PUBLISHERS,35,9925756414902312,Online access :,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-SOCI.310.002-33903A PEOPLE'S HISTORY OF...
3,NO GOOD - 1 COPY,SPRING 2027,MUSICOLOGY,202720-MUCO.380.001-23480,"Gay, Leslie",9781319327569,A POCKET STYLE MANUAL,DIANA HACKER; NANCY SOMMERS,MACMILLAN HIGHER EDUCATION,20,9926216362902312,Single User Single User,Electronic version at ProQuest; ProQuest Ebook...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-MUCO.380.001-23480A POCKET STYLE MANUAL
4,GOOD,SPRING 2027,INTERDISCIPLIN EARLY CHILDHOOD,202720-IEC.471.002-39593,"Caudle, Lori",9781138495180,A PRACTICAL GUIDE TO ACTION RESEARCH AND TEACH...,"INCE, AMANDA AND KITTO, ELEANOR",TAYLOR & FRANCIS GROUP,10,9926083718402312,NaN,Electronic version at Taylor & Francis; Taylor...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-IEC.471.002-39593A PRACTICAL GUIDE TO A...
5,GOOD,SPRING 2027,ENGLISH,202720-ENGL.355.003-25830,"Saenkhum, Tanita",9780809326631,APPEALS IN MODERN RHETORIC : AN ORDINARY LANGU...,"KILLINGSWORTH, M. JIMMIE",SOUTHERN ILLINOIS UNIVERSITY PRESS,25,9926062446102312,Unlimited Access Unlimited Access,Electronic version at Project Muse; Project Mu...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ENGL.355.003-25830APPEALS IN MODERN RHE...
6,GOOD,SPRING 2027,ARTIFICIAL INTELLEGIENCE,202720-AI.311.001-38951,"Sahin, Burak",9781805123552,ARTIFICIAL INTELLIGENCE FOR CYBERSECURITY,"KOLOSNJAJI, BOJAN, XU, PENG, ZARRAS, APOSTOLIS...","PACKT PUBLISHING, LIMITED",50,9926914606002312,NaN,Electronic version at ProQuest; O'Reilly for H...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-AI.311.001-38951ARTIFICIAL INTELLIGENCE...
7,NO GOOD - 1 COPY,SPRING 2027,ENGLISH,202720-ENGL.253.002-20648,"Newburn, Harry",9780307388629,BELOVED,"MORRISON, TONI",KNOPF DOUBLEDAY PUBLISHING GROUP,25,9925919468902312,Online Access - 1 user,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ENGL.253.002-20648BELOVED
8,NO GOOD - BLOCKED SITE,SPRING 2027,FOOD SCIENCE,202720-FDSC.210.001-37487,"Sommer, Stephan",9780306472749,BREWING,"YOUNG, TOM W. AND LEWIS, MICHAEL J.",SPRINGER,45,9925584156402312,NaN,Electronic version at Springer Nature; Springe...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-FDSC.210.001-37487BREWING
9,GOOD,SPRING 2027,ENTREPRENEURSHIP,202720-ENT.451.001-28877,"Mel?ndez, Eduardo",9781118173626,BUSINESS MODEL GENERATION : A HANDBOOK FOR VIS...,"OSTERWALDER, ALEXANDER AND PIGNEUR, YVES","WILEY & SONS, INCORPORATED, JOHN",57,9926913209102312,NaN,Electronic version at ProQuest; O'Reilly for H...,https://utk.primo.exlibrisgroup.com/permalink/...,202720-ENT.451.001-28877BUSINESS MODEL GENERAT...


In [88]:
# Use this for the first list of the semester only
#newChecks = bookstoreExport

# Use this for every list except for the first of the semester
newChecks = bookstoreExport[~bookstoreExport['Section Title Code'].isin(alreadyChecked['Section Title Code'])]

newChecks

,index,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity,MMS ID,Access_Type,Electronic Availability,Permalink,Section Title Code
127,127,SPRING 2027,INDUSTRIAL ENGINEERING,202720-IE.201.001-30782,"Null, Gary",9781603581486,NaN,NaN,NaN,46,9926479564302311,,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...,NaN


In [55]:
# If there is nothing in newChecks, skip this cell

newChecks.to_excel('Bookstore Access Check ' +date_str+'.xlsx')

### After checking all of the books, save the file as "Bookstore Access Check YYYY-MM-DD.xlsx" to be loaded into the cells below.

## 14. Export the Bookstore Results to Send to the Bookstore

In [89]:
# Use this for the first list of the semester only
#finalChecked = pd.read_excel(r'Bookstore Access Check '+date_str+'.xlsx')

# Use this for every list except for the first of the semester
previousChecked1 = pd.read_excel(r'Bookstore Access Check 2026-11-20.xlsx')

finalChecked = pd.concat([previousChecked1])

In [90]:
# Use this for the first list of the semester only
#finalExport = bookstoreExport

# Use this for every list except for the first of the semester
finalExport = finalChecked[finalChecked['Access Check'] == 'GOOD']

finalExport = finalExport[['Academic Term', 'Academic Department Name', 'Section Code', 'Instructor Name', 'ISBN', 'Short Title', 'Book Author', 'Book Publisher', 'Enrollment Capacity', 'MMS ID', 'Access_Type', 'Electronic Availability', 'Permalink']]

finalExport = finalExport.sort_values(by = 'Short Title')

finalExport

,Academic Term,Academic Department Name,Section Code,Instructor Name,ISBN,Short Title,Book Author,Book Publisher,Enrollment Capacity,MMS ID,Access_Type,Electronic Availability,Permalink
2,SPRING 2027,SOCIOLOGY,202720-SOCI.310.002-33903,"Lowery, Grady",9781317325291,A PEOPLE'S HISTORY OF THE UNITED STATES,"ZINN, HOWARD",HARPERCOLLINS PUBLISHERS,35,9925756414902312,Online access :,Electronic version at EBSCOhost; EBSCO Ebooks ...,https://utk.primo.exlibrisgroup.com/permalink/...
4,SPRING 2027,INTERDISCIPLIN EARLY CHILDHOOD,202720-IEC.471.002-39593,"Caudle, Lori",9781138495180,A PRACTICAL GUIDE TO ACTION RESEARCH AND TEACH...,"INCE, AMANDA AND KITTO, ELEANOR",TAYLOR & FRANCIS GROUP,10,9926083718402312,NaN,Electronic version at Taylor & Francis; Taylor...,https://utk.primo.exlibrisgroup.com/permalink/...
5,SPRING 2027,ENGLISH,202720-ENGL.355.003-25830,"Saenkhum, Tanita",9780809326631,APPEALS IN MODERN RHETORIC : AN ORDINARY LANGU...,"KILLINGSWORTH, M. JIMMIE",SOUTHERN ILLINOIS UNIVERSITY PRESS,25,9926062446102312,Unlimited Access Unlimited Access,Electronic version at Project Muse; Project Mu...,https://utk.primo.exlibrisgroup.com/permalink/...
6,SPRING 2027,ARTIFICIAL INTELLEGIENCE,202720-AI.311.001-38951,"Sahin, Burak",9781805123552,ARTIFICIAL INTELLIGENCE FOR CYBERSECURITY,"KOLOSNJAJI, BOJAN, XU, PENG, ZARRAS, APOSTOLIS...","PACKT PUBLISHING, LIMITED",50,9926914606002312,NaN,Electronic version at ProQuest; O'Reilly for H...,https://utk.primo.exlibrisgroup.com/permalink/...
9,SPRING 2027,ENTREPRENEURSHIP,202720-ENT.451.001-28877,"Mel?ndez, Eduardo",9781118173626,BUSINESS MODEL GENERATION : A HANDBOOK FOR VIS...,"OSTERWALDER, ALEXANDER AND PIGNEUR, YVES","WILEY & SONS, INCORPORATED, JOHN",57,9926913209102312,NaN,Electronic version at ProQuest; O'Reilly for H...,https://utk.primo.exlibrisgroup.com/permalink/...
10,SPRING 2027,EDUCATION OF THE DEAF AND HARD OF HEARING,202720-EDDE.425.004-36357,"Skyer, Michael",9781944838195,"CASE STUDIES IN DEAF EDUCATION : INQUIRY, APPL...","GUARDINO, CAROLINE, BEAL, JENNIFER S., CANNON,...",GALLAUDET UNIVERSITY PRESS,25,9926431786402312,Unlimited Access Unlimited Access,Electronic version at JSTOR; JSTOR Ebooks (JST...,https://utk.primo.exlibrisgroup.com/permalink/...
11,SPRING 2027,EDUCATION OF THE DEAF AND HARD OF HEARING,202720-EDDE.425.005-41804,"Skyer, Michael",9781944838195,"CASE STUDIES IN DEAF EDUCATION : INQUIRY, APPL...","GUARDINO, CAROLINE, BEAL, JENNIFER S., CANNON,...",GALLAUDET UNIVERSITY PRESS,25,9926431786402312,Unlimited Access Unlimited Access,Electronic version at JSTOR; JSTOR Ebooks (JST...,https://utk.primo.exlibrisgroup.com/permalink/...
12,SPRING 2027,CONSTRUCTION SCIENCE & MANAGEMENT,202720-CSM.454.001-38345,"Smith, Angela",9781605955995,CONSTRUCTION SITE EROSION AND SEDIMENT CONTROL...,"PITT, ROBERT, CLARK, SHIRLEY, AND LAKE, DONALD","DE STECH PUBLICATIONS, INCORPORATED",60,9926398687202312,NaN,Electronic version at Knovel; Knovel Civil Eng...,https://utk.primo.exlibrisgroup.com/permalink/...
13,SPRING 2027,CONSTRUCTION SCIENCE & MANAGEMENT,202720-CSM.454.002-40986,"Saal, Lauren",9781605955995,CONSTRUCTION SITE EROSION AND SEDIMENT CONTROL...,"PITT, ROBERT, CLARK, SHIRLEY, AND LAKE, DONALD","DE STECH PUBLICATIONS, INCORPORATED",100,9926398687202312,NaN,Electronic version at Knovel; Knovel Civil Eng...,https://utk.primo.exlibrisgroup.com/permalink/...
20,SPRING 2027,JOURNALISM AND MEDIA,202720-JMED.490.001-38425,"Martinez, Michael",9781040394885,CRITICAL PHOTOJOURNALISM : CONTEMPORARY ETHICS...,"PIXLEY, TARA AND WALGREN, JUDY","ROUTLEDGE, CHAPMAN & HALL, INCORPORATED",20,9926800030302312,NaN,Electronic version at Taylor & Francis; Taylor...,https://utk.primo.exlibrisgroup.com/permalink/...


In [91]:
finalExport.to_excel(date_str+'BookStoreResults_TPQ.xlsx', index = False) #Send this to the Bookstore